In [1]:
# ==============================================================================
# Step 1: Import Required Libraries
# ==============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import root_mean_squared_error, r2_score

# ==============================================================================
# Step 2: Load and Prepare Dataset
# ==============================================================================
data = fetch_california_housing(as_frame=True)
df = pd.concat([data.data, data.target.rename("HousePrice")], axis=1)

X = df.drop("HousePrice", axis=1)
y = df["HousePrice"]

# ==============================================================================
# Step 3 & 4: Train-Test Split and Feature Scaling
# (Scaling after split prevents data leakage from test to train)
# ==============================================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Scaled full features for cross-validation reference
X_scaled = scaler.fit_transform(X)

# ==============================================================================
# Step 5: Detect Overfitting (Default Unconstrained Decision Tree)
# ==============================================================================
default_tree = DecisionTreeRegressor(random_state=42)
default_tree.fit(X_train_scaled, y_train)

train_pred = default_tree.predict(X_train_scaled)
test_pred = default_tree.predict(X_test_scaled)

train_rmse = root_mean_squared_error(y_train, train_pred)
test_rmse = root_mean_squared_error(y_test, test_pred)

print(f"Default Tree - Train RMSE: {train_rmse:.4f}")
print(f"Default Tree - Test RMSE:  {test_rmse:.4f}")
# A near-zero train RMSE alongside high test RMSE confirms severe overfitting.

# ==============================================================================
# Step 6: Baseline Models & Cross-Validation
# ==============================================================================
# 1. Linear Regression Baseline
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
lr_test_pred = lr.predict(X_test_scaled)
lr_rmse = root_mean_squared_error(y_test, lr_test_pred)
lr_r2 = r2_score(y_test, lr_test_pred)
lr_cv = -cross_val_score(lr, X_train_scaled, y_train, scoring="neg_root_mean_squared_error", cv=5).mean()

# 2. Ridge Regression Baseline (from Task-2)
ridge = Ridge(alpha=1.0, random_state=42)
ridge.fit(X_train_scaled, y_train)
ridge_test_pred = ridge.predict(X_test_scaled)
ridge_rmse = root_mean_squared_error(y_test, ridge_test_pred)
ridge_r2 = r2_score(y_test, ridge_test_pred)
ridge_cv = -cross_val_score(ridge, X_train_scaled, y_train, scoring="neg_root_mean_squared_error", cv=5).mean()

# ==============================================================================
# Step 7: Hyperparameter Tuning via GridSearchCV
# ==============================================================================
param_grid = {
    "max_depth": [3, 5, 7, 10, 12],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 4, 8]
}

grid = GridSearchCV(
    estimator=DecisionTreeRegressor(random_state=42),
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1
)
grid.fit(X_train_scaled, y_train)

print("Best Hyperparameters:", grid.best_params_)
tuned_cv_rmse = -grid.best_score_

# ==============================================================================
# Step 8: Evaluate Optimized Model
# ==============================================================================
best_tree = grid.best_estimator_

best_tree_train_pred = best_tree.predict(X_train_scaled)
best_tree_test_pred = best_tree.predict(X_test_scaled)

tuned_train_rmse = root_mean_squared_error(y_train, best_tree_train_pred)
tuned_test_rmse = root_mean_squared_error(y_test, best_tree_test_pred)
tuned_r2 = r2_score(y_test, best_tree_test_pred)

print(f"Tuned Tree - Train RMSE: {tuned_train_rmse:.4f}")
print(f"Tuned Tree - Test RMSE:  {tuned_test_rmse:.4f}")
print(f"Tuned Tree - R2 Score:   {tuned_r2:.4f}")

# ==============================================================================
# Step 9: Model Comparison Summary Table
# ==============================================================================
results = {
    "Model": [
        "Linear Regression",
        "Ridge Regression",
        "Default Decision Tree",
        "Tuned Decision Tree"
    ],
    "Train RMSE": [
        root_mean_squared_error(y_train, lr.predict(X_train_scaled)),
        root_mean_squared_error(y_train, ridge.predict(X_train_scaled)),
        train_rmse,
        tuned_train_rmse
    ],
    "5-Fold CV RMSE": [
        lr_cv,
        ridge_cv,
        -cross_val_score(default_tree, X_train_scaled, y_train, scoring="neg_root_mean_squared_error", cv=5).mean(),
        tuned_cv_rmse
    ],
    "Test RMSE": [
        lr_rmse,
        ridge_rmse,
        test_rmse,
        tuned_test_rmse
    ],
    "Test R2 Score": [
        lr_r2,
        ridge_r2,
        r2_score(y_test, test_pred),
        tuned_r2
    ]
}

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

Default Tree - Train RMSE: 0.0000
Default Tree - Test RMSE:  0.7021
Best Hyperparameters: {'max_depth': 12, 'min_samples_leaf': 8, 'min_samples_split': 20}
Tuned Tree - Train RMSE: 0.4574
Tuned Tree - Test RMSE:  0.6017
Tuned Tree - R2 Score:   0.7238
                Model   Train RMSE  5-Fold CV RMSE  Test RMSE  Test R2 Score
    Linear Regression 7.196757e-01        0.720527   0.745581       0.575788
     Ridge Regression 7.196758e-01        0.720527   0.745557       0.575816
Default Decision Tree 3.017601e-16        0.724663   0.702131       0.623791
  Tuned Decision Tree 4.573506e-01        0.609849   0.601658       0.723756
